In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.metrics import(
    accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix,roc_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [3]:
df=pd.read_csv('banking_loan_default_dataset_500_rows.csv')

In [4]:
df.head()

,customer_id,age,income_monthly,loan_amount,credit_score,employment_years,debt_to_income,existing_loans,dependents,bank_balance,interest_rate,employment_status,marital_status,education,loan_purpose,city_tier,previous_defaults,loan_default
0,CUST1362,29,143036.0,200322,574.0,4.7,0.22,5,4,301629,13.12,Business,Married,Graduate,Home,Tier 3,0,No
1,CUST1074,28,56102.0,863738,810.0,2.8,0.13,5,3,620324,8.92,Salaried,Married,Undergraduate,Home,Tier 2,0,Yes
2,CUST1375,40,93703.0,428165,687.0,8.3,0.51,3,2,94037,10.44,Business,Divorced,Undergraduate,Education,Tier 3,0,Yes
3,CUST1156,52,123787.0,364997,651.0,0.6,0.17,0,0,483454,8.08,Self-employed,Married,Postgraduate,Education,Tier 3,1,Yes
4,CUST1105,25,48355.0,855688,662.0,3.6,0.47,3,0,906311,8.91,Salaried,Married,Undergraduate,Education,Tier 1,0,Yes


In [5]:
df.shape

(500, 18)

In [6]:
df.info

<bound method DataFrame.info of     customer_id  age  income_monthly  loan_amount  credit_score  \
0      CUST1362   29        143036.0       200322         574.0   
1      CUST1074   28         56102.0       863738         810.0   
2      CUST1375   40         93703.0       428165         687.0   
3      CUST1156   52        123787.0       364997         651.0   
4      CUST1105   25         48355.0       855688         662.0   
..          ...  ...             ...          ...           ...   
495    CUST1107   59        174542.0       150158         636.0   
496    CUST1271   23        122573.0       677433         557.0   
497    CUST1349   60         19062.0       502268           NaN   
498    CUST1436   49        134748.0       201563         664.0   
499    CUST1103   55             NaN       698373         717.0   

     employment_years  debt_to_income  existing_loans  dependents  \
0                 4.7            0.22               5           4   
1                 2.8    

In [7]:
df.isnull().sum().sort_values(ascending=False).head(15)

credit_score         15
income_monthly       15
education            15
debt_to_income       15
employment_years     15
customer_id           0
age                   0
loan_amount           0
dependents            0
existing_loans        0
bank_balance          0
interest_rate         0
employment_status     0
marital_status        0
loan_purpose          0
dtype: int64

In [8]:
target_column='loan_default'
df=df.dropna(subset=[target_column]).copy()

X=df.drop(columns=[target_column])
y=df[target_column]

print('Target Values : ')
print(y.value_counts())

Target Values : 
loan_default
Yes    357
No     143
Name: count, dtype: int64


In [9]:
label_encoder=LabelEncoder()
y=label_encoder.fit_transform(y.astype(str))

print('Classes : ',list(label_encoder.classes_))
print('Encoded classes : ',np.unique(y))

Classes :  ['No', 'Yes']
Encoded classes :  [0 1]


In [10]:
numeric_features=X.select_dtypes(include=np.number).columns.tolist()
categorical_features=X.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric features : ',numeric_features)
print('\nCategorical features : ',categorical_features)


Numeric features :  ['age', 'income_monthly', 'loan_amount', 'credit_score', 'employment_years', 'debt_to_income', 'existing_loans', 'dependents', 'bank_balance', 'interest_rate', 'previous_defaults']

Categorical features :  ['customer_id', 'employment_status', 'marital_status', 'education', 'loan_purpose', 'city_tier']


In [11]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.20,random_state=42,
    stratify=y
)
print('Training data : ',X_train.shape)
print('Testing data : ',X_test.shape)


Training data :  (400, 17)
Testing data :  (100, 17)


In [13]:
numeric_transformer=Pipeline([
    ('imputer',SimpleImputer(strategy='median'))
])
categorical_transformer=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])
preprocessor=ColumnTransformer([
    ('num',numeric_transformer,numeric_features),
    ('cat',categorical_transformer,categorical_features)
])

In [15]:
xgb_model=XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',xgb_model)
])
xgb_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['customer_id','age','income_monthly',...,'loan_purpose','city_tier', 'previous_defaults']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [16]:
lgb_model=LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)
lgb_pipeline=Pipeline([
    ('preprecessor',preprocessor),
    ('model',lgb_model)
])

lgb_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprecessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['customer_id','age','income_monthly',...,'loan_purpose','city_tier', 'previous_defaults']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [22]:
xgb_pred=xgb_pipeline.predict(X_test)
xgb_prob=xgb_pipeline.predict_proba(X_test)[:,1]

lgb_pred=lgb_pipeline.predict(X_test)
lgb_prob=lgb_pipeline.predict_proba(X_test)[:,1]

def get_metrics(y_true,y_pred,y_prob):
    return{
        'Accuracy': accuracy_score(y_true,y_pred),
        'Precision': precision_score(y_true,y_pred,zero_division=0),
        'Recall':recall_score(y_true,y_pred,zero_division=0),
        'f1_score':f1_score(y_true,y_pred,zero_division=0),
        'ROC-AUC': roc_auc_score(y_true,y_prob)
    }
results=pd.DataFrame([
    get_metrics(y_test,xgb_pred,xgb_prob),
    get_metrics(y_test,lgb_pred,lgb_prob)
],index=['XGBoost','LightGBM'])

results.round(4)    

,Accuracy,Precision,Recall,f1_score,ROC-AUC
XGBoost,0.72,0.7471,0.9155,0.8228,0.7115
LightGBM,0.69,0.7439,0.8592,0.7974,0.7067
